In [1]:
import faiss
import pandas as pd
import numpy as np
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity
from ast import literal_eval
from tqdm import tqdm
import requests
import seaborn as sns
import matplotlib.pyplot as plt
from collections import Counter
from tqdm import tqdm
from scipy.stats import zscore, entropy
import json, os
import Levenshtein
from IPython.display import display, HTML
import difflib, html
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np

In [2]:
def highlight_diff(a,b,c1='background:salmon',c2='background:lightgreen'):
  s=difflib.SequenceMatcher(None,a,b)
  h1=[]; h2=[]
  for tag,i1,i2,j1,j2 in s.get_opcodes():
    if tag=='equal':
      h1.append(html.escape(a[i1:i2])); h2.append(html.escape(b[j1:j2]))
    elif tag=='replace':
      h1.append(f"<span style='{c1}'>"+html.escape(a[i1:i2])+"</span>")
      h2.append(f"<span style='{c2}'>"+html.escape(b[j1:j2])+"</span>")
    elif tag=='delete':
      h1.append(f"<span style='{c1}'>"+html.escape(a[i1:i2])+"</span>")
    elif tag=='insert':
      h2.append(f"<span style='{c2}'>"+html.escape(b[j1:j2])+"</span>")
  out=f"<div style='font-family:monospace;white-space:pre-wrap'>"+''.join(h1)+"</div><div style='font-family:monospace;white-space:pre-wrap;margin-top:4px'>"+''.join(h2)+"</div>"
  display(HTML(out))

def get_debate_context(hitloc, tk):
    return tk[pd.to_datetime(tk.date) == hitloc['tk_date']]

def plot_scatter_grid(
    df,
    p_id_highlight=None,
    highlight_word=None,
    highlight_color="yellow",
    n_cols=15
):

    df = df.reset_index(drop=True)

    N = len(df)
    n_rows = int(np.ceil(N / n_cols))

    # --- Colors (party) ---
    parties = df["party-ref"].fillna("UNKNOWN").unique()
    cmap = plt.get_cmap("tab10")
    party_colors = {p: cmap(i % 10) for i, p in enumerate(parties)}

    # --- Markers (role) ---
    marker_list = ["o", "s", "^", "D", "P", "X", "*", "v", "<", ">"]
    roles = df["role"].fillna("UNKNOWN").unique()
    role_markers = {r: marker_list[i % len(marker_list)] for i, r in enumerate(roles)}

    # --- Grid positions ---
    xs = np.arange(N) % n_cols
    ys = np.arange(N) // n_cols
    ys = -ys  # invert so 0 is top row

    fig, ax = plt.subplots(figsize=(6,12))

    # Normalize highlight word for comparison
    word = highlight_word.lower() if highlight_word else None

    # --- Plot scatters ---
    for i, row in df.iterrows():
        x, y = xs[i], ys[i]
        base_color = party_colors[row["party-ref"]]
        marker = role_markers[row["role"]]

        # --- Word highlight ---
        contains_word = False
        if word:
            txt = str(row["text"]).lower()
            contains_word = word.lower() in txt

        # --- Category: word highlight takes precedence over party color ---
        if contains_word:
            color = base_color
            size = 150
            edge = "black"
            width = 1.2,
            ls = 'dashed'

        # --- p_id highlight (if not already highlighted by word) ---
        elif row["p_id"] == p_id_highlight:
            color = base_color
            size = 300
            edge = "black"
            width = 1.5
            ls='solid'

        else:
            color = base_color
            size = 50
            edge = "none"
            width = 0.5
            ls='solid'

        ax.scatter(
            x, y,
            s=size,
            c=[color],
            marker=marker,
            edgecolor=edge,
            linewidth=width,
            linestyle=ls
        )

    # --- Aesthetics ---
    ax.set_xlim(-0.5, n_cols - 0.5)
    ax.set_ylim(-(n_rows), 0.5)
    ax.set_xticks(range(n_cols))
    ax.set_yticks([])

    ax.set_title(
        "Speech Grid — Color = Party, Marker = Role\n"
        "(Highlighted p_id and word matches)"
    )

    # --- Legends ---
    # Party legend
    party_handles = [
        plt.Line2D([0], [0], marker='o', color='white',
                   markerfacecolor=party_colors[p], markersize=10,
                   linestyle='') for p in parties
    ]
    party_legend = ax.legend(party_handles, parties, title="Party", loc="upper right")
    ax.add_artist(party_legend)

    # Role legend
    role_handles = [
        plt.Line2D([0], [0], marker=role_markers[r], color='black',
                   linestyle='', markersize=10) for r in roles
    ]
    ax.legend(role_handles, roles, title="Role", loc="lower right")

    plt.tight_layout()
    plt.show()

In [3]:
tk = pd.read_csv("data/tk.csv",sep='\t')

/tmp/ipykernel_373429/3443076796.py:1: DtypeWarning: Columns (2,3) have mixed types. Specify dtype option on import or set low_memory=False.
  tk = pd.read_csv("data/tk.csv",sep='\t')


In [4]:
with open('data/meta-reports.json', 'r') as f:
    report_ids = json.load(f)
    report_mtd = {rid['_id']:rid for rid in report_ids}

In [5]:
def word_in_date(word, tk):
    results = {}

    for date, df in tk.groupby(pd.to_datetime(tk.date)):
        tx = " ".join(df.text).lower()
        results[date] = True if word in tx else False
    return results

def get_activiteit_by_date(date: pd.Timestamp) -> dict:
    """
    Fetch the first plenair debat activity for a given date from the Tweede Kamer OData API.

    Args:
        date (pd.Timestamp): The date to filter on.

    Returns:
        dict: The first activity result as a dictionary, or None if no results.
    """
    # Build URL with date filter
    base_url = "https://gegevensmagazijn.tweedekamer.nl/OData/v4/2.0/Activiteit"
    date_filter = (
        f"year(Datum) eq {date.year} "
        f"and month(Datum) eq {date.month} "
        f"and day(Datum) eq {date.day}"
    )

    odata_filter = f"contains(soort,'plenair debat') and Status eq 'Uitgevoerd' and {date_filter}"

    # Encode URL
    url = f"{base_url}?$filter={requests.utils.quote(odata_filter)}&$count=true&expand=Document"

    # Request API
    response = requests.get(url)
    response.raise_for_status()
    data = response.json()

    # Return first result if exists
    if data.get("value"):
        return data["value"][0]
    return None

org_in_date = word_in_date("rekenkamer", tk)

In [6]:
dfm = pd.read_csv('results/matches-rep-kst-sentences.csv', parse_dates=['rep_date','tk_date'])
dfm['lvs'] = [Levenshtein.ratio(s,t) for s,t in tqdm(zip(dfm.rep_sentence, dfm.tk_sentence))]
dfm['dydiff'] = (dfm.tk_date - dfm.rep_date ).dt.days
# dfm['court_in_debate'] = dfm.tk_date.map(org_in_date)
dfm['sem_syn_diff'] = dfm.score - dfm.lvs


1285035it [00:00, 1310099.06it/s]


In [37]:
dfms = dfm[
        (dfm.score > .75) &
        (dfm.dydiff > 0) &
        (dfm.dydiff < 365) &
        (dfm.lvs > .5) &
        # (dfm.tk_sentence != dfm.rep_sentence)
        (dfm.court_in_debate == False) &
        ~(dfm.tk_sentence.str.contains('ORG'))
         ]

print(dfms.shape)

(94, 21)


In [38]:
# dfms = dfms[["tk_sentence","rep_sentence","score","lvs"]]
dfms['scoren'] = (dfms.score + dfms.lvs) / 2

/tmp/ipykernel_373429/2053667576.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfms['scoren'] = (dfms.score + dfms.lvs) / 2


In [40]:
dfms[(dfms.tk_sentence.str[0].str.isalpha()) & ~(dfms.tk_sentence.str.contains("antwoord",case=False)) & ~(dfms.rep_id.str.contains('beantwoord'))]

,score,rep_id,rep_date,rep_detail_url,rep_sentence,rep_sent_id,rep_sid,tk_id,tk_source,tk_date,...,tk_source_title,tk_source_subject,tk_sentence,tk_sent_id,tk_sid,lvs,dydiff,sem_syn_diff,court_in_debate,scoren
36501,0.759153,rekenkamer__kamerstuk:2013:05:03:actualisering...,2013-05-03,https://www.rekenkamer.nl/publicaties/kamerstu...,Op 16 april 2013 heeft het Europees Parlement ...,9,rekenkamer__kamerstuk:2013:05:03:actualisering...,kamerstukken__brief_regering:274b4bad-bfaf-44a...,kamerstukken__brief_regering,2014-03-11,...,JBZ-Raad,Uitvoering moties ten aanzien van de richtlijn...,Het Europees Parlement heeft op 27 februari 20...,3,kamerstukken__brief_regering:274b4bad-bfaf-44a...,0.523490,312,0.235663,False,0.641321
38107,0.757705,rekenkamer__kamerstuk:2013:05:30:actualisering...,2013-05-30,https://www.rekenkamer.nl/publicaties/kamerstu...,Een afschrift van deze brief sturen wij aan de...,27,rekenkamer__kamerstuk:2013:05:30:actualisering...,kamerstukken__brief_regering:e5651248-f737-487...,kamerstukken__brief_regering,2014-04-10,...,Toekomst financiële sector,Fiscale behandeling vermogensinstrumenten banken,Een afschrift van deze brief is verzonden aan ...,29,kamerstukken__brief_regering:e5651248-f737-487...,0.770053,315,-0.012348,False,0.763879
42923,0.759636,rekenkamer__kamerstuk:2013:09:26:aandachtspunt...,2013-09-26,https://www.rekenkamer.nl/publicaties/kamerstu...,Een afschrift van deze brief sturen wij naar d...,79,rekenkamer__kamerstuk:2013:09:26:aandachtspunt...,kamerstukken__brief_regering:e5651248-f737-487...,kamerstukken__brief_regering,2014-04-10,...,Toekomst financiële sector,Fiscale behandeling vermogensinstrumenten banken,Een afschrift van deze brief is verzonden aan ...,29,kamerstukken__brief_regering:e5651248-f737-487...,0.739583,196,0.020052,False,0.749610
49643,0.785724,rekenkamer__kamerstuk:2013:11:07:input-voor-ro...,2013-11-07,https://www.rekenkamer.nl/publicaties/kamerstu...,"Een geharmoniseerde methodologie, heldere defi...",73,rekenkamer__kamerstuk:2013:11:07:input-voor-ro...,kamerstukken__brief_regering:f7cc817c-7b3b-423...,kamerstukken__brief_regering,2014-02-13,...,Verbetering verantwoording en begroting,Aandachtspunten van Algemene Rekenkamer over E...,Eenduidige verslaggevingsregels vormen een bel...,12,kamerstukken__brief_regering:f7cc817c-7b3b-423...,0.752577,98,0.033147,False,0.769151
53067,0.815644,rekenkamer__kamerstuk:2013:11:28:europees-econ...,2013-11-28,https://www.rekenkamer.nl/publicaties/kamerstu...,Deze nota wordt vervolgens voor advies voorgel...,63,rekenkamer__kamerstuk:2013:11:28:europees-econ...,kamerstukken__schriftelijke_vragen_antwoord:b1...,kamerstukken__schriftelijke_vragen_antwoord,2014-01-23,...,NaN,Antwoord op vragen van het lid Schouw over het...,Vervolgens is het ontwerpbesluit voorgelegd aa...,5,kamerstukken__schriftelijke_vragen_antwoord:b1...,0.781250,56,0.034394,False,0.798447
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1021934,0.752288,rekenkamer__rapport:2020:10:13:verzekerd-van-z...,2020-10-13,https://www.rekenkamer.nl/publicaties/rapporte...,Het Zorginstituut zelf is terughoudend met de ...,22,rekenkamer__rapport:2020:10:13:verzekerd-van-z...,kamerstukken__algemeen_overleg:f27c4186-8f78-4...,kamerstukken__algemeen_overleg,2020-12-03,...,Kwaliteit van zorg,"Verslag van een algemeen overleg, gehouden op ...",Het Zorginstituut zelf is terughoudend gebleke...,43,kamerstukken__algemeen_overleg:f27c4186-8f78-4...,0.729064,51,0.023224,False,0.740676
1023894,0.779127,rekenkamer__rapport:2020:11:02:focus-op-digita...,2020-11-02,https://www.rekenkamer.nl/publicaties/publicat...,Zie bijvoorbeeld het KIM-onderzoek Thuiswerken...,105,rekenkamer__rapport:2020:11:02:focus-op-digita...,kamerstukken__brief_regering:fc2e9b70-4908-49e...,kamerstukken__brief_regering,2021-02-24,...,Infectieziektenbestrijding,Reactie op het RIVM-onderzoek i.v.m. thuiswerk...,"Een overzicht van studies naar de omvang, bele...",94,kamerstukken__brief_rege

In [41]:
dfms.loc[49643]['tk_sentence'],dfms.loc[49643]['rep_sentence']

('Eenduidige verslaggevingsregels vormen een belangrijke randvoor waarde voor onderling vergelijkbare en betrouwbare gegevens op zowel nationaal als Europees niveau.',
 'Een geharmoniseerde methodologie, heldere definities en een eenduidig boekhoudstelsel en verslaggevingsregels vormen randvoorwaarden voor onderling vergelijkbare en betrouwbare gegevens op zowel nationaal als Europees niveau.')

In [44]:
dfms.loc[49643]['tk_detail_url']

'https://www.tweedekamer.nl/kamerstukken/detail?id=2014Z02795&did=2014D05560'

In [16]:
# Write to Markdown file
with open("/home/rb/Documents/Code/Rekenkamer/results/formatted_matches_rep_tk.md", "w", encoding="utf-8") as f:
    for i, r in dfms.sort_values('score').iterrows():
        try:
            f.write('---\n\n')
            f.write(f'### HIT — SCORE: **{round(r["score"], 2)}\n**')
            f.write(f'### HIT — LVS: **{round(r["lvs"], 2)}**\n')

            f.write(f'**REPORT SENTENCE:** {r['rep_sentence']}\n')
            f.write(f'**TK SENTENCE:** {r['tk_sentence']}\n\n')

            rp_mtd = report_mtd.get(r['rep_id'])

            # Report info
            f.write(f'**Report Title:** {rp_mtd.get("title")}\n')
            f.write(f'**Report Date:** {rp_mtd.get("date")}\n')
            f.write(f'**Report Department:** {rp_mtd.get("category")}\n\n')

            # Debate info
            f.write(f'**Debate Date:** {str(r['tk_date'])}\n\n')

        except Exception as e:
            continue


# Macro

In [20]:
dfs = dfm[dfm.score>.75].sort_values("score",ascending=False)

In [21]:
dfs = dfs[~dfs.tk_id.str.contains("schriftel|brief")]

In [23]:
dfs = dfs.drop_duplicates("rep_sid",keep="first")

In [24]:
dfs.rep_id.value_counts()

rep_id
rekenkamer__rapport:2014:05:21:resultaten-verantwoordingsonderzoek-2013-bij-het-ministerie-van-defensie                                                              8
rekenkamer__rapport:2012:05:16:rapport-bij-het-jaarverslag-2011-van-het-ministerie-van-defensie                                                                      6
rekenkamer__rapport:2018:05:16:resultaten-verantwoordingsonderzoek-2017-bij-het-ministerie-van-binnenlandse-zaken-en-koninkrijksrelaties                             6
rekenkamer__rapport:2019:05:15:resultaten-verantwoordingsonderzoek-2018-ministerie-van-defensie                                                                      6
rekenkamer__kamerstuk:2017:06:13:beantwoording-vragen-tweede-kamer-over-de-resultaten-verantwoordingsonderzoek-2016-bij-het-ministerie-van-bzk                       5
                                                                                                                                                              